# 🥇 Silver → Gold v2

Builds two Gold tables:
1. `gold.mhcet_cutoffs` — main app table: cutoffs per college+branch+category+gender across all seat pools
2. `gold.mhcet_cutoffs_by_pool` — detailed: same but split by seat pool (State/HomeUniv/Other etc.)

**Cutoff definition:** minimum MHT-CET score allotted in that round for that combination  
**Likely round:** first round where student's score ≥ cutoff (computed at query time in app)

## Cell 1 — Config

In [ ]:
CATALOG       = 'rankrangers_project_data'
SILVER_TABLE  = f'{CATALOG}.silver.mhcet_allotments'
GOLD_TABLE    = f'{CATALOG}.gold.mhcet_cutoffs'
GOLD_POOL     = f'{CATALOG}.gold.mhcet_cutoffs_by_pool'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold')
print(f'Source : {SILVER_TABLE}')
print(f'Target : {GOLD_TABLE}')
print(f'Target : {GOLD_POOL}')

## Cell 2 — Load Silver & Preview

In [ ]:
from pyspark.sql import functions as F

df = spark.table(SILVER_TABLE)
print(f'Silver rows: {df.count():,}')

# Exclude non-student-facing pools from main cutoff table
# AI = All India (JEE score, different eligibility)
# Minority = requires specific community
# Orphan = special category
EXCLUDE_POOLS = ['AllIndia', 'Minority', 'Orphan']

df_main = df.filter(~F.col('seat_pool').isin(EXCLUDE_POOLS))
print(f'After excluding AI/Minority/Orphan: {df_main.count():,}')

## Cell 3 — Build Main Gold Table
One row per `college + branch + category + gender`.  
Takes MIN score across all seat pools (State + HomeUniv + Other) — most accessible cutoff for the student.

In [ ]:
from pyspark.sql import functions as F

df_gold = df_main.groupBy(
    'institute_code',
    'institute_name',
    'branch_name',
    'clean_category',
    'seat_gender',
    'is_ews',
    'is_tfws'
).agg(
    # Min score per round = cutoff (floor — lowest score that got a seat)
    F.round(F.min(F.when(F.col('cap_round_num') == 1, F.col('mhtcet_score'))), 4).alias('cap1_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 2, F.col('mhtcet_score'))), 4).alias('cap2_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 3, F.col('mhtcet_score'))), 4).alias('cap3_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 4, F.col('mhtcet_score'))), 4).alias('cap4_cutoff'),

    # Max score per round = top of range
    F.round(F.max(F.when(F.col('cap_round_num') == 1, F.col('mhtcet_score'))), 4).alias('cap1_max'),
    F.round(F.max(F.when(F.col('cap_round_num') == 2, F.col('mhtcet_score'))), 4).alias('cap2_max'),
    F.round(F.max(F.when(F.col('cap_round_num') == 3, F.col('mhtcet_score'))), 4).alias('cap3_max'),
    F.round(F.max(F.when(F.col('cap_round_num') == 4, F.col('mhtcet_score'))), 4).alias('cap4_max'),

    # Total seats filled across all rounds
    F.count('*').alias('total_seats_filled'),

    # Seats per round
    F.count(F.when(F.col('cap_round_num') == 1, 1)).alias('cap1_seats'),
    F.count(F.when(F.col('cap_round_num') == 2, 1)).alias('cap2_seats'),
    F.count(F.when(F.col('cap_round_num') == 3, 1)).alias('cap3_seats'),
    F.count(F.when(F.col('cap_round_num') == 4, 1)).alias('cap4_seats'),

    # Pools available for this combo
    F.collect_set('seat_pool').alias('seat_pools_available')
)

# Only keep combos that had seats in CAP-IV (final picture)
df_gold = df_gold.filter(F.col('cap4_cutoff').isNotNull())

print(f'Gold rows: {df_gold.count():,}')
display(df_gold.orderBy('institute_name', 'branch_name').limit(10))

## Cell 4 — Write Main Gold Table

In [ ]:
df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', True) \
    .saveAsTable(GOLD_TABLE)

print(f'✅ {GOLD_TABLE}: {spark.table(GOLD_TABLE).count():,} rows')

## Cell 5 — Build Pool-Level Gold Table
Same as main but split by `seat_pool` — for deeper analysis.

In [ ]:
df_pool = df.groupBy(
    'institute_code',
    'institute_name',
    'branch_name',
    'clean_category',
    'seat_gender',
    'seat_pool'
).agg(
    F.round(F.min(F.when(F.col('cap_round_num') == 1, F.col('mhtcet_score'))), 4).alias('cap1_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 2, F.col('mhtcet_score'))), 4).alias('cap2_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 3, F.col('mhtcet_score'))), 4).alias('cap3_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 4, F.col('mhtcet_score'))), 4).alias('cap4_cutoff'),
    F.count('*').alias('total_seats_filled'),
).filter(F.col('cap4_cutoff').isNotNull())

df_pool.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', True) \
    .saveAsTable(GOLD_POOL)

print(f'✅ {GOLD_POOL}: {spark.table(GOLD_POOL).count():,} rows')

## Cell 6 — Save Dropdown Metadata for Streamlit

In [ ]:
import json

df_g = spark.table(GOLD_TABLE)

branches   = sorted([r[0] for r in df_g.select('branch_name').distinct().collect()])
categories = sorted([r[0] for r in df_g.select('clean_category').distinct().collect()])

meta = {
    'branches':   branches,
    'categories': categories
}

meta_path = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/app_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'✅ Metadata saved: {meta_path}')
print(f'   Branches   : {len(branches)}')
print(f'   Categories : {len(categories)} → {categories}')

## Cell 7 — Simulate App Query
Exactly what Streamlit will run when a student submits their inputs.

In [ ]:
from pyspark.sql import functions as F

# ── Student inputs ────────────────────────────────────────────────────────────
student_score    = 85.5
student_category = 'OBC'
student_gender   = 'M'       # M = male → GOBCS, GOPENS etc.
student_branch   = 'Computer Science and Engineering'

# ── Query Gold ────────────────────────────────────────────────────────────────
df_result = spark.table(GOLD_TABLE).filter(
    (F.col('clean_category') == student_category) &
    (F.col('seat_gender').isin(student_gender, 'ANY')) &
    (F.col('branch_name')    == student_branch) &
    (F.col('cap4_cutoff')    <= student_score)     # must be reachable in final round
).withColumn(
    'likely_round',
    F.when(F.col('cap1_cutoff') <= student_score, 'CAP-I')
     .when(F.col('cap2_cutoff') <= student_score, 'CAP-II')
     .when(F.col('cap3_cutoff') <= student_score, 'CAP-III')
     .when(F.col('cap4_cutoff') <= student_score, 'CAP-IV')
     .otherwise('Unlikely')
).withColumn(
    'score_gap',
    F.round(student_score - F.col('cap1_cutoff'), 2)
).select(
    'institute_name',
    'branch_name',
    'clean_category',
    'cap1_cutoff',
    'cap2_cutoff',
    'cap3_cutoff',
    'cap4_cutoff',
    'likely_round',
    'score_gap',
    'total_seats_filled'
).orderBy('cap1_cutoff', ascending=False)

count = df_result.count()
print(f'Student: {student_branch} | {student_category} | {"Male" if student_gender=="M" else "Female"} | Score: {student_score}')
print(f'Colleges in range: {count}')
display(df_result)